# 生成并显示彩色点云

RGB-D 图像同时提供颜色和深度信息。本节将每个有效像素反投影为相机坐标系中的三维点，并把对应 RGB 像素的颜色附加到三维点上，最终生成、显示并保存彩色点云。


## 1. 彩色点云是什么？

点云是一组离散的三维点，而不是普通图片，也不是带三角形连接关系的网格。一个彩色点可以写成：

$$
\mathrm{point}_i = (x_i, y_i, z_i, r_i, g_i, b_i)
$$

其中 $(x_i, y_i, z_i)$ 是点在某个坐标系中的位置，$(r_i, g_i, b_i)$ 是颜色。深度图中的每一个有效像素，原则上都可以生成一个三维点，所以点云密度通常与深度图分辨率相关。

图像坐标写成 $(u,v)=(横坐标,纵坐标)$，而数组索引写成 $[行,列]$，因此同一个像素在代码中读取为 `image[v, u]`；彩色图像通常写成 `image[v, u, :]`。

## 2. 一个像素如何变成一个彩色三维点？

假设 RGB 和深度已经对齐，像素坐标为 $(u, v)$，深度图中的有效深度为 $Z_c$，相机内参为 $f_x, f_y, c_x, c_y$。使用针孔相机模型可以得到：

$$
x = \frac{(u-c_x)Z_c}{f_x}, \qquad
y = \frac{(v-c_y)Z_c}{f_y}, \qquad
z = Z_c
$$

然后从 RGB 图像中取出同一个像素的颜色：

$$
\mathrm{color}(u,v) = \mathrm{RGB}[v,u]
$$

$$
\mathrm{point}(u,v) = (x, y, z, \mathrm{RGB}[v,u])
$$

所以一个点的完整来源是：

- 深度图中的 $D(v,u)$ 和像素坐标 $(u,v)$ 决定三维位置 $(x,y,z)$；
- RGB 图中的 $\mathrm{RGB}[v,u]$ 决定点的颜色 $(r,g,b)$。

这里的深度默认是相机坐标系中的 $Z_c$，即沿光轴方向的深度，不是相机光心到点的欧氏距离。

## 3. 真实 RGB-D 数据的关键细节

### 3.1 RGB 和深度为什么需要对齐

RGB 相机和深度相机通常有不同的光心、内参、分辨率和镜头畸变。因此，RGB 和深度图中相同的数组下标，默认不代表同一个空间位置。只有在已经完成空间配准时，`RGB[v, u]` 和 `Depth[v, u]` 才能直接组成一个彩色三维点。分辨率相同也不等于已经对齐；没有对齐时，颜色会在物体边缘发生错位。

对齐的本质不是 resize，而是把一个相机中的像素恢复成三维点，再投影到另一个相机的像素平面。设下标 $d$ 表示深度相机，$c$ 表示 RGB 相机。两台相机分别有内参 $K_d$、$K_c$，并用外参描述深度相机坐标系到 RGB 相机坐标系的变换：

$$
P_d=Z_dK_d^{-1}
\begin{bmatrix}u_d\\v_d\\1\end{bmatrix},
\qquad
P_c=R_{cd}P_d+t_{cd}
$$

最后使用 RGB 相机内参投影：

$$
\begin{bmatrix}\tilde u_c\\\tilde v_c\\\tilde w_c\end{bmatrix}=K_cP_c,
\qquad
u_c=\frac{\tilde u_c}{\tilde w_c},
\quad
v_c=\frac{\tilde v_c}{\tilde w_c}
$$

完整链路为：

```text
深度像素 (u_d, v_d) + 深度 Z_d
          ↓ K_d^-1 反投影
深度相机坐标 P_d
          ↓ R_cd, t_cd
RGB 相机坐标 P_c
          ↓ K_c 投影和透视除法
RGB 像素 (u_c, v_c)
```

因此，对齐实际上是“像素 → 三维点 → 另一个像素”的几何对应。实际工程中还要确认 $R_{cd},t_{cd}$ 的方向；如果手里的是 RGB 相机到深度相机的外参，就不能直接代入上面的公式，而要先取逆。

### 3.2 对齐到哪个坐标系

常见的结果有两种：

- **深度对齐到 RGB**：以 RGB 图像的分辨率为目标，把每个深度点投影到 RGB 网格。这样可以直接使用 `aligned_depth[v, u]` 和 `RGB[v, u]`。
- **RGB 对齐到深度**：以深度图的分辨率为目标，把颜色重新采样到深度网格。这样适合遍历有效深度像素生成点云。

两种结果都可以称为对齐，但输出数组的分辨率、像素坐标系和后续使用的内参不同，代码中必须明确目标坐标系。

### 3.3 投影时的冲突、空洞与遮挡

投影后可能有多个深度像素落到同一个 RGB 像素，也可能有 RGB 像素没有对应的深度值。前者通常使用 z-buffer，只保留在目标相机坐标系中更近的点；后者可以保持为无效值，也可以根据应用需求插值，但插值得到的是估计值，不是新的传感器测量。两个相机的光心不重合，所以遮挡边缘处即使标定准确，也可能出现彩边或小范围错位。

TUM 数据集中的 RGB 和深度通常是两个文件序列，需要先按时间戳匹配，再使用数据集提供的相机参数和深度尺度。

### 3.4 深度尺度和单位

深度 PNG 经常以 `uint16` 保存，像素值不一定就是米：

```python
depth_m = depth_raw.astype(np.float32) / depth_scale
```

深度以毫米保存时常用 `depth_scale = 1000`；TUM RGB-D 常见格式使用 `depth_scale = 5000`。尺度错了，点云整体尺寸也会错。

### 3.5 无效深度

0、NaN、无穷大和超出可信范围的深度不能反投影：

```python
valid = np.isfinite(depth) & (depth > 0) & (depth < depth_trunc)
```

深度空洞常见于黑色、反光、透明表面、物体边缘和过远区域。点云是传感器从一个视角得到的离散采样，并不保证覆盖物体完整表面。

## 4. 还需要知道的成像误差

### 镜头畸变

反投影公式默认针孔相机。真实镜头存在径向和切向畸变，因此应使用去畸变后的图像和内参，或者在反投影中显式加入畸变模型。入门实验可以先使用 SDK 或数据集已经校正、对齐的 RGB-D 图像。

### 遮挡与时间不同步

RGB 相机和深度相机看到的内容可能因视差、遮挡或时间差而不同。运动物体尤其容易出现彩色边缘和深度边缘不一致。

### 内参必须匹配

`fx, fy, cx, cy` 必须对应生成深度坐标的相机和图像分辨率。缩放图像后，内参也要按相同比例缩放。

## 5. Open3D 的坐标系

Open3D 从 RGB-D 生成的点云通常位于相机坐标系中，常见约定是：

```text
X：向右
Y：向下
Z：向前
```

这和图像坐标方向一致，但与一些 3D 软件中 Y 向上的习惯不同。为了显示得更像常见的 3D 场景，可以做显示用变换：

```python
pcd.transform([[1, 0, 0, 0],
               [0, -1, 0, 0],
               [0, 0, -1, 0],
               [0, 0, 0, 1]])
```

这个变换只改变坐标表示和显示方向，不能修复错误的内参、深度尺度或 RGB-D 对齐。保存点云前要明确自己保存的是相机坐标，还是变换后的显示坐标。

## 6. 体素下采样

体素是三维空间中的小立方体。`voxel_size = 0.02` 表示边长为 2 厘米的网格。如果多个点落入同一个体素，Open3D 会把它们合并成一个代表点，位置和颜色通常取平均。

下采样可以减少点数、让密度更均匀并降低计算量，但会牺牲细节。`voxel_size` 越大，点越少、细节损失越多；它的单位必须和点云坐标单位一致。

## 最终链路：

```text
RGB 图 + 深度图
       ↓ 检查时间、分辨率和空间对齐
有效像素 (u, v) + 深度 Zc
       ↓ K^-1 反投影
相机坐标点 (x, y, z)
       ↓ 取 RGB[v, u] 的颜色
彩色点云
       ↓ 体素网格聚合
下采样彩色点云 / PLY 文件
```

## Open3D 实验代码

本实验使用 TUM RGB-D 的官方 PNG 序列生成彩色点云。TUM 发布的深度图已经由 OpenNI 预注册到 RGB 图像，因此先按时间戳匹配 RGB/深度，再使用 TUM 的深度尺度和相机内参。

代码按下面的顺序运行：

1. 导入库并设置数据集参数。
2. 读取 `rgb.txt`、`depth.txt`，按时间戳组成 `frames`。
3. 读取一帧 RGB-D，检查图像尺寸。
4. 用 RGB-D 生成相机坐标系下的单帧彩色点云。
5. 对单帧点云下采样并显示。
6. 读取每帧位姿，把多帧点云变换到世界坐标系后融合。
7. 把融合结果复制一份，转换到第一帧相机视角后显示。

In [ ]:
# 1. 导入库
from pathlib import Path
import numpy as np
import open3d as o3d

# 2. 设置相机、数据集和实验参数
# TUM Freiburg 1 数据集的相机内参。
TUM_FR1_INTRINSIC = np.array(
    [[517.3, 0.0, 318.6], [0.0, 516.5, 255.3], [0.0, 0.0, 1.0]],
    dtype=np.float64,
)
# TUM 深度图中的数值除以 5000 后，才是以米为单位的深度。
TUM_DEPTH_SCALE = 5000.0
data_dir = Path("data/tum/rgbd_dataset_freiburg1_xyz")
# 这里的索引不是原始文件名编号，而是时间戳匹配后列表中的位置。
frame_index = 78
# RGB 和深度图的时间戳差不超过这个值时，才认为它们属于同一帧。
# 当前序列的最大时间差约为 0.02016 秒，因此留出少量余量。
max_timestamp_delta = 0.021
# 只使用不超过这个距离的深度值，单位是米。
depth_trunc = 4.0


### 1. 读取 TUM 索引文件

`rgb.txt` 和 `depth.txt` 保存了时间戳与图像路径。先把它们读取成 Python 列表。

In [ ]:
def read_tum_index(index_path, data_dir):
    """读取一个 TUM 索引文件，返回时间戳和图像路径。"""
    entries = []
    for line in index_path.read_text().splitlines():
        fields = line.split()
        # 以 “#” 开头的是说明文字，不是图像记录。
        if len(fields) < 2 or fields[0].startswith("#"):
            continue
        timestamp = float(fields[0])
        image_path = data_dir / fields[1]
        entries.append((timestamp, image_path))
    return sorted(entries)

### 2. 按时间戳匹配 RGB 和深度图

当前 TUM 序列的两份索引已经按时间顺序一一对应。代码按顺序配对，并用时间戳差检查配对是否可靠。

In [ ]:
def discover_tum_frames(data_dir, max_timestamp_delta):
    """按顺序配对 RGB 和深度图，并检查时间戳差。"""
    data_dir = Path(data_dir)
    rgb_entries = read_tum_index(data_dir / "rgb.txt", data_dir)
    depth_entries = read_tum_index(data_dir / "depth.txt", data_dir)

    # 这个 TUM 序列的 RGB 和深度索引已经按时间排序，且一一对应。
    # 因此按顺序 zip 比不断推进一个共享 depth_index 更可靠。
    if len(rgb_entries) != len(depth_entries):
        raise ValueError("RGB 和深度记录数不一致，不能按顺序配对")

    frames = []
    for frame_id, (rgb_entry, depth_entry) in enumerate(
        zip(rgb_entries, depth_entries)
    ):
        color_timestamp, color_path = rgb_entry
        depth_timestamp, depth_path = depth_entry
        timestamp_delta = abs(depth_timestamp - color_timestamp)

        if timestamp_delta > max_timestamp_delta:
            continue

        frames.append({
            "frame_id": frame_id,
            "timestamp": (color_timestamp + depth_timestamp) / 2.0,
            "color_path": color_path,
            "depth_path": depth_path,
        })

    if not frames:
        raise ValueError("没有找到可匹配的 TUM RGB-D 帧")
    return frames

In [ ]:
# 读取索引并完成 RGB、深度图的时间戳匹配。
frames = discover_tum_frames(data_dir, max_timestamp_delta)

# 选择一帧用于单帧实验；frame_index 是匹配成功后的列表索引。
if frame_index >= len(frames):
    raise IndexError("帧索引超出范围")
example_frame = frames[frame_index]
example_frame_id = example_frame["frame_id"]

print("匹配帧数:", len(frames))
print("当前帧:", example_frame_id)
print("时间戳:", example_frame["timestamp"])
print("RGB 路径:", example_frame["color_path"])
print("深度路径:", example_frame["depth_path"])

### 3. 读取一帧 RGB 图和深度图

读取后先检查尺寸是否一致。只有同一个像素确实对应同一个空间位置，才能把颜色贴到三维点上。

```txt
深度图像素
    ↓ 深度相机内参反投影
深度相机坐标系中的三维点 P_d
    ↓ 外参变换 R、t
RGB 相机坐标系中的三维点 P_rgb
    ↓ RGB 相机内参投影
RGB 图像平面中的像素坐标 (u_rgb, v_rgb)
```

In [ ]:
def load_frame_images(frame):
    """读取一帧 RGB 图像和深度图，并检查分辨率。"""
    color = np.asarray(o3d.io.read_image(str(frame["color_path"])))
    depth = np.asarray(o3d.io.read_image(str(frame["depth_path"])))
    if color.ndim != 3 or color.shape[2] != 3:
        raise ValueError("RGB 图像必须是三通道图像")
    if color.shape[:2] != depth.shape:
        raise ValueError("RGB 和深度分辨率不一致")
    return color, depth

In [ ]:
# color_np 是三通道颜色数组，depth_raw 是原始深度值数组。
color_np, depth_raw = load_frame_images(example_frame)

print("RGB 形状:", color_np.shape, "数据类型:", color_np.dtype)
print("深度形状:", depth_raw.shape, "数据类型:", depth_raw.dtype)
print("有效深度像素:", int(np.count_nonzero(depth_raw)))

### 4. 从 RGB-D 图像生成彩色点云

这里把深度像素反投影到相机坐标系，再把 RGB 图中同位置的颜色附加到三维点。具体计算由 Open3D 完成。

#### 数学公式与代码的区别

从数学上看，像素反投影主要使用焦距和主点：

```text
x = (u - cx) * z / fx
y = (v - cy) * z / fy
```

因此，真正参与公式计算的核心内参是 `fx`、`fy`、`cx`、`cy`。但是在代码中，Open3D 还要求提供 `width` 和 `height`，因为它需要知道图像尺寸，以及这组内参对应哪一个分辨率。

可以这样理解：`fx`、`fy`、`cx`、`cy` 负责三维坐标计算；`width`、`height` 负责描述完整的相机模型和检查图像尺寸是否匹配。如果图像缩放了，图像宽高和相机内参通常都要一起缩放。

In [ ]:
def create_point_cloud(color, depth, depth_trunc=4.0):
    """使用相机内参，把 RGB-D 图像转换为彩色点云。"""
    # depth 的形状是 (height, width)，因此可以直接得到图像尺寸。
    height, width = depth.shape

    # 创建针孔相机模型。这个对象告诉 Open3D：相机如何把三维点投影到图像像素。
    # width、height 是图像宽高，必须和当前 RGB、深度图的尺寸一致。
    intrinsic = o3d.camera.PinholeCameraIntrinsic(
        width,
        height,
        # 内参矩阵第一行第一列：水平方向焦距 fx。
        TUM_FR1_INTRINSIC[0, 0],
        # 内参矩阵第二行第二列：垂直方向焦距 fy。
        TUM_FR1_INTRINSIC[1, 1],
        # 内参矩阵第一行第三列：主点横坐标 cx。
        TUM_FR1_INTRINSIC[0, 2],
        # 内参矩阵第二行第三列：主点纵坐标 cy。
        TUM_FR1_INTRINSIC[1, 2],
    )

    # 把 NumPy 数组包装成 Open3D 能识别的 RGBD 图像。
    # RGB 图提供颜色，深度图提供每个像素到相机的距离。
    rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
        # 把颜色数组转换为 Open3D 图像对象。
        o3d.geometry.Image(color),
        # 把深度数组转换为 Open3D 图像对象。
        o3d.geometry.Image(depth),
        # 原始深度值除以 5000 才是米，所以 TUM 使用 5000。
        depth_scale=TUM_DEPTH_SCALE,
        # 只使用不超过这个距离的深度值，单位是米。
        depth_trunc=depth_trunc,
        # False 表示保留 RGB 颜色，不把彩色图转换成灰度图。
        convert_rgb_to_intensity=False,
    )

    # Open3D 根据 RGBD 图像和相机内参完成反投影：
    # 有效深度像素 -> 相机坐标系中的三维点，并附上对应 RGB 颜色。
    return o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, intrinsic)

# -----------------------------------------------------------------------------
# 下面的函数只负责把点云显示出来，不参与 RGB-D 到三维点的计算。
# -----------------------------------------------------------------------------
def show_point_cloud_in_notebook(pcd, title, max_points=50_000, camera_view=False):
    """用 Plotly 在 Notebook 中预览点云，可选择相机朝向。"""
    import plotly.graph_objects as go

    points = np.asarray(pcd.points)
    colors = np.asarray(pcd.colors)
    if len(points) == 0:
        raise ValueError("点云为空，无法显示")

    # Plotly 只显示均匀抽取的一部分点；原始 pcd 不会被修改。
    count = min(len(points), max_points)
    indices = np.linspace(0, len(points) - 1, count, dtype=int)
    shown_points = points[indices]

    if len(colors) == len(points):
        rgb = np.rint(np.clip(colors[indices], 0.0, 1.0) * 255).astype(np.uint8)
        marker_colors = [f"rgb({r},{g},{b})" for r, g, b in rgb]
    else:
        marker_colors = "#4c78a8"

    # Plotly 的相机向场景中心看。相机坐标系中 z 轴向前、y 轴向下，
    # 所以从负 z 侧看向中心，并把页面向上方向设为 -y。
    scene = {"aspectmode": "data", "dragmode": "orbit"}
    if camera_view:
        scene["camera"] = {
            "eye": {"x": 0.0, "y": 0.0, "z": -1.8},
            "up": {"x": 0.0, "y": -1.0, "z": 0.0},
            "projection": {"type": "perspective"},
        }

    figure = go.Figure(go.Scatter3d(
        x=shown_points[:, 0],
        y=shown_points[:, 1],
        z=shown_points[:, 2],
        mode="markers",
        marker={"size": 1.5, "color": marker_colors, "opacity": 0.9},
    ))
    figure.update_layout(
        title=title,
        height=680,
        margin={"l": 0, "r": 0, "b": 0, "t": 40},
        scene=scene,
    )
    figure.show()


# 这是可选的二维重投影检查，不是多帧融合的必要步骤。
def show_camera_projection_in_notebook(pcd, title, image_shape):
    """将三维点云重投影为二维 RGB 对齐检查图，不是二维点云。"""
    import plotly.graph_objects as go

    height, width = image_shape
    points = np.asarray(pcd.points)
    colors = np.asarray(pcd.colors)
    if len(points) == 0:
        raise ValueError("点云为空，无法显示")

    # 相机坐标中的 z 必须为正，才能投影到 RGB 图像平面。
    valid = np.isfinite(points).all(axis=1) & (points[:, 2] > 0)
    camera_points = points[valid]
    if len(camera_points) == 0:
        raise ValueError("相机前方没有可显示的点")

    fx, fy = TUM_FR1_INTRINSIC[0, 0], TUM_FR1_INTRINSIC[1, 1]
    cx, cy = TUM_FR1_INTRINSIC[0, 2], TUM_FR1_INTRINSIC[1, 2]
    depth = camera_points[:, 2]
    u = np.rint(fx * camera_points[:, 0] / depth + cx).astype(np.int64)
    v = np.rint(fy * camera_points[:, 1] / depth + cy).astype(np.int64)

    inside = (u >= 0) & (u < width) & (v >= 0) & (v < height)
    u, v, depth = u[inside], v[inside], depth[inside]
    if len(u) == 0:
        raise ValueError("没有点落在 RGB 图像范围内")

    if len(colors) == len(points):
        rgb = np.rint(np.clip(colors[valid][inside], 0.0, 1.0) * 255).astype(np.uint8)
    else:
        rgb = np.full((len(u), 3), [76, 120, 168], dtype=np.uint8)

    # 同一像素可能有多个三维点：按深度排序后，仅保留最近点（z-buffer）。
    flat_pixel = v * width + u
    order = np.lexsort((depth, flat_pixel))
    sorted_pixel = flat_pixel[order]
    nearest = np.r_[True, sorted_pixel[1:] != sorted_pixel[:-1]]
    selected = order[nearest]

    image = np.zeros((height, width, 3), dtype=np.uint8)
    image[v[selected], u[selected]] = rgb[selected]

    figure = go.Figure(go.Image(z=image))
    figure.update_layout(
        title=title,
        height=680,
        margin={"l": 0, "r": 0, "b": 0, "t": 40},
    )
    figure.update_xaxes(title="RGB 像素 u")
    figure.update_yaxes(title="RGB 像素 v", autorange="reversed", scaleanchor="x")
    figure.show()

In [ ]:
# 使用相机内参把每个有效深度像素转换成三维点，并保留对应颜色。
pcd = create_point_cloud(color_np, depth_raw, depth_trunc)

print("原始点数:", len(pcd.points))

In [ ]:
# 设置体素边长为 0.02 米，也就是 2 厘米
# voxel_size 的单位必须和点云坐标单位一致
voxel_size = 0.02

# 按体素网格合并点云中落入同一体素的点
# 通常会用体素内点的平均位置和平均颜色作为代表点
downsampled = pcd.voxel_down_sample(voxel_size)

# 统计下采样后的点数量
print("原始点数:", len(pcd.points))
print("下采样后点数:", len(downsampled.points))

# 这仍是三维彩色点云，只是将初始观察方向设为 RGB 相机朝向。
show_point_cloud_in_notebook(
    downsampled,
    title="原始彩色点云（RGB 相机朝向的 3D 预览）",
    camera_view=True,
)


### 6. 定义位姿变换辅助函数

多帧融合需要两件事：先找到某一帧对应的真值位姿，再把位姿记录转换成 Open3D 使用的 4×4 变换矩阵。把这两步单独写成函数后，下面的融合循环只保留主流程。

In [ ]:
def find_nearest_pose(timestamp, groundtruth):
    """返回与图像时间戳最接近的一条真值位姿。"""
    pose_index = np.abs(groundtruth[:, 0] - timestamp).argmin()
    return groundtruth[pose_index]


def pose_to_world_camera_transform(pose):
    """把 TUM 位姿记录转换为“相机坐标系 -> 世界坐标系”矩阵。"""
    _, tx, ty, tz, qx, qy, qz, qw = pose

    transform = np.eye(4)
    transform[:3, :3] = o3d.geometry.get_rotation_matrix_from_quaternion(
        [qw, qx, qy, qz]
    )
    transform[:3, 3] = [tx, ty, tz]
    return transform


def transform_point_cloud_to_first_frame_view(pcd, first_frame, groundtruth):
    """复制世界坐标系点云，并转换到第一帧相机坐标系。"""
    first_pose = find_nearest_pose(first_frame["timestamp"], groundtruth)
    first_T_world_camera = pose_to_world_camera_transform(first_pose)

    # 不修改原始融合结果，只转换一份副本用于观察。
    first_frame_pcd = o3d.geometry.PointCloud(pcd)
    first_frame_pcd.transform(np.linalg.inv(first_T_world_camera))
    return first_frame_pcd


## 7. 用 Open3D 融合多帧点云

单帧点云位于各自的相机坐标系，不能直接堆叠。这个数据集提供了 `groundtruth.txt`，其中保存每个时刻的相机真值位姿；下面直接使用 Open3D 把每帧点云变换到世界坐标系，追加到同一个融合点云，最后再下采样。

下面会遍历 `frames` 中的全部已匹配帧。为了便于学习，融合循环保留在主代码单元中；位姿查找和矩阵构造则放到上一个辅助函数单元。

### 多帧点云到底如何合并？

**关键不是把多张点云直接相加，而是先统一坐标系。**每帧反投影出的点云都以当时相机为原点；相机移动后，直接堆叠会把不同地点的场景错误地放在同一个原点，造成重影。

```text
第 i 帧 RGB-D  →  相机坐标点 P_Ci  →  用位姿 T_WCi 变到世界坐标 P_W
第 j 帧 RGB-D  →  相机坐标点 P_Cj  →  用位姿 T_WCj 变到世界坐标 P_W
                                                    ↓
                                      将世界坐标点追加为 fused_pcd
                                                    ↓
                                      体素下采样，合并重叠和过密点
```

`T_world_camera` 是 4×4 齐次变换矩阵，包含旋转 `R` 和平移 `t`：`P_W = T_WC · P_C`。TUM 的 `groundtruth.txt` 直接提供每帧相机在固定世界坐标系中的位置和四元数旋转；代码将四元数转为 `R`，组成 `T_world_camera`。

| 代码 | 实际含义 |
| --- | --- |
| `frame_pcd.transform(T_world_camera)` | 将当前帧从自己的相机坐标系摆到世界中的正确位置与方向。 |
| `fused_pcd += frame_pcd` | 追加已经对齐的点和颜色；这一步不寻找对应点，也不做平均。 |
| `voxel_down_sample(0.02)` | 用边长 2 cm 的三维网格分桶；同一体素中的点生成一个代表点，位置与颜色取平均。 |

代码先对**每帧**下采样以控制内存；对整个 `fused_pcd` 再下采样，才会消除不同帧在同一位置产生的重复点。融合结束后，`fused_pcd` 保持在世界坐标系中。

这里假设环境静止。运动的人、手或物体会出现重影；位姿有误差也会使墙面或桌边变厚。真实机器人没有 `groundtruth.txt` 时，需要用 RGB-D 里程计、ICP 或 SLAM 估计同样的 `T_world_camera`。

In [ ]:
# 读取所有相机真值位姿；每行格式：timestamp tx ty tz qx qy qz qw
groundtruth = np.loadtxt(data_dir / "groundtruth.txt")

# 这个体素大小同时用于“每帧下采样”和最后的全局下采样。
voxel_size = 0.005  # 单位米

# 融合结果始终保存在世界坐标系中。
fused_pcd = o3d.geometry.PointCloud()
fused_frame_count = 0

# 每次循环只处理一帧，顺序就是：位姿 -> RGB-D -> 点云 -> 世界坐标 -> 追加。
for frame in frames:
    # 1. 找到当前帧位姿，并构造相机坐标系到世界坐标系的变换。
    pose = find_nearest_pose(frame["timestamp"], groundtruth)
    T_world_camera = pose_to_world_camera_transform(pose)

    # 2. 读取当前帧的 RGB 和深度图。
    color, depth = load_frame_images(frame)

    # 3. 根据 RGB-D 生成当前帧的相机坐标系点云。
    frame_pcd = create_point_cloud(color, depth, depth_trunc)

    # 4. 先减少当前帧点数，再进行坐标变换。
    frame_pcd = frame_pcd.voxel_down_sample(voxel_size)

    # 5. 把当前帧从相机坐标系变换到世界坐标系。
    frame_pcd.transform(T_world_camera)

    # 6. 追加到总点云；此时所有点已经位于同一个世界坐标系。
    fused_pcd += frame_pcd
    fused_frame_count += 1

# 7. 对所有帧合并后的点云再下采样，合并重叠区域的重复点。
fused_pcd = fused_pcd.voxel_down_sample(voxel_size)
print(f"已融合 {fused_frame_count} 帧，最终点数：{len(fused_pcd.points)}")

# 复制并转换一份点云，从第一帧相机视角观察。
fused_from_first_camera = transform_point_cloud_to_first_frame_view(
    fused_pcd,
    frames[0],
    groundtruth,
)
show_point_cloud_in_notebook(
    fused_from_first_camera,
    title="多帧融合彩色点云（第一帧相机视角）",
    max_points=150_000,
    camera_view=True,
)
